# Evaluación Parcial 3 — Programación para la Ciencia de Datos

## Análisis interactivo de emisiones de CO₂

**Tema del trabajo:** analizar la evolución histórica de emisiones de CO₂, comparar países y revisar la relación entre población y emisiones usando Python, Pandas, Plotly y un mini dashboard en notebook.

**Dataset utilizado:** `owid-co2-data.csv`.

**Integrantes:** completar nombres del grupo.

## 1. Pregunta de análisis

Siguiendo la idea de que una buena visualización debe responder una pregunta, este trabajo se organiza en torno a cuatro preguntas simples:

1. ¿Cómo han evolucionado las emisiones mundiales de CO₂ desde 1990?
2. ¿Qué países concentran más emisiones en el año más reciente disponible?
3. ¿Existe relación entre población y emisiones totales de CO₂?
4. ¿Cómo se comporta Chile frente a otros países seleccionados de la región?

## 2. Importar librerías

Se usa **Pandas** para el manejo de datos y **Plotly Express** para crear visualizaciones interactivas con hover, zoom y exploración visual.

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

## 3. Cargar datos

En Google Colab, primero sube el archivo `owid-co2-data.csv` al panel lateral de archivos. Luego ejecuta esta celda.

In [ ]:
# Nombre del archivo CSV usado en el proyecto.
ruta_csv = "owid-co2-data.csv"

# Carga de datos.
df = pd.read_csv(ruta_csv)

# Vista inicial del dataset.
df.head()

## 4. Exploración inicial

Antes de graficar, revisamos tamaño, columnas y tipos de datos. Esto permite entender qué variables tenemos disponibles y qué limpieza básica será necesaria.

In [ ]:
print("Filas y columnas:", df.shape)
print("\nColumnas disponibles:")
print(df.columns.tolist())

In [ ]:
# Resumen de columnas principales para el análisis.
df[["country", "year", "iso_code", "population", "gdp", "co2", "co2_per_capita", "share_global_co2"]].info()

In [ ]:
# Estadísticas descriptivas de variables numéricas relevantes.
df[["year", "population", "gdp", "co2", "co2_per_capita", "share_global_co2"]].describe()

## 5. Preparación y limpieza básica

Para mantener el análisis claro se seleccionan solo las columnas necesarias. Además:

- Se convierten variables numéricas con `pd.to_numeric`.
- Se separan países reales de agregados como `World`, usando `iso_code`.
- Se eliminan valores vacíos solo cuando la visualización necesita esa variable.

In [ ]:
columnas = [
    "country", "year", "iso_code", "population", "gdp",
    "co2", "co2_per_capita", "share_global_co2"
]

datos = df[columnas].copy()

datos["year"] = pd.to_numeric(datos["year"], errors="coerce")

for columna in ["population", "gdp", "co2", "co2_per_capita", "share_global_co2"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

# Países reales: tienen iso_code. Agregados como World quedan separados.
paises = datos[datos["iso_code"].notna()].copy()
mundo = datos[datos["country"] == "World"].dropna(subset=["co2"]).copy()

anio_reciente = int(paises["year"].max())
paises_reciente = paises[paises["year"] == anio_reciente].copy()

print("Año más reciente disponible:", anio_reciente)
print("Países/registros en el último año:", paises_reciente.shape[0])

## 6. Mini dashboard: indicadores principales

Un dashboard debe guiar la mirada del usuario. Primero mostramos indicadores simples y luego los gráficos que explican esos indicadores.

In [ ]:
# Cálculo de indicadores principales.
co2_mundo = mundo.loc[mundo["year"] == anio_reciente, "co2"].iloc[0]
co2_mundo_1990 = mundo.loc[mundo["year"] == 1990, "co2"].iloc[0]
crecimiento_mundo = (co2_mundo / co2_mundo_1990 - 1) * 100

chile_reciente = paises_reciente[paises_reciente["country"] == "Chile"].iloc[0]
co2_chile = chile_reciente["co2"]
co2_chile_pc = chile_reciente["co2_per_capita"]

html = f"""
<div style="font-family:Arial; display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin:10px 0 20px 0;">
  <div style="padding:16px; border-radius:14px; background:#eef6ff; border:1px solid #bdd7ff;">
    <div style="color:#64748b; font-size:13px;">Año analizado</div>
    <div style="font-size:26px; font-weight:700; color:#173B63;">{anio_reciente}</div>
  </div>
  <div style="padding:16px; border-radius:14px; background:#eef6ff; border:1px solid #bdd7ff;">
    <div style="color:#64748b; font-size:13px;">CO₂ mundial</div>
    <div style="font-size:26px; font-weight:700; color:#173B63;">{co2_mundo:,.0f} Mt</div>
  </div>
  <div style="padding:16px; border-radius:14px; background:#eef6ff; border:1px solid #bdd7ff;">
    <div style="color:#64748b; font-size:13px;">CO₂ Chile</div>
    <div style="font-size:26px; font-weight:700; color:#173B63;">{co2_chile:.1f} Mt</div>
  </div>
  <div style="padding:16px; border-radius:14px; background:#eef6ff; border:1px solid #bdd7ff;">
    <div style="color:#64748b; font-size:13px;">Chile per cápita</div>
    <div style="font-size:26px; font-weight:700; color:#173B63;">{co2_chile_pc:.2f} t</div>
  </div>
</div>
"""

display(HTML(html))
print(f"Entre 1990 y {anio_reciente}, las emisiones mundiales crecieron aproximadamente {crecimiento_mundo:.1f}%.")

## 7. Visualización 1 — Gráfico de líneas

**Pregunta:** ¿Cómo han evolucionado las emisiones mundiales de CO₂ desde 1990?

El gráfico de líneas es adecuado porque permite ver una tendencia a través del tiempo.

In [ ]:
mundo_1990 = mundo[(mundo["year"] >= 1990) & (mundo["year"] <= anio_reciente)].copy()

fig_lineas = px.line(
    mundo_1990,
    x="year",
    y="co2",
    markers=True,
    title=f"Evolución mundial de emisiones de CO₂, 1990-{anio_reciente}",
    labels={"year": "Año", "co2": "CO₂ total (millones de toneladas)"}
)

fig_lineas.update_layout(template="plotly_white", title_x=0.02)
fig_lineas.show()

**Interpretación:** el gráfico muestra una tendencia general al alza desde 1990. Aunque existen caídas en algunos años, el valor más reciente es superior al de 1990, por lo que la lectura principal es que las emisiones mundiales han aumentado en el período analizado.

## 8. Visualización 2 — Gráfico de barras

**Pregunta:** ¿Qué países tienen mayores emisiones totales en el año más reciente?

El gráfico de barras es útil para comparar categorías, en este caso países.

In [ ]:
top10_co2 = (
    paises_reciente
    .dropna(subset=["co2"])
    .sort_values("co2", ascending=False)
    .head(10)
)

fig_barras = px.bar(
    top10_co2,
    x="country",
    y="co2",
    color="country",
    title=f"Top 10 países por emisiones de CO₂ en {anio_reciente}",
    labels={"country": "País", "co2": "CO₂ total (millones de toneladas)"},
    hover_data=["co2_per_capita", "population"]
)

fig_barras.update_layout(template="plotly_white", title_x=0.02, showlegend=False)
fig_barras.show()

**Interpretación:** el gráfico permite identificar rápidamente qué países concentran más emisiones totales. La comparación por barras facilita ver diferencias de magnitud entre países.

## 9. Visualización 3 — Gráfico de dispersión

**Pregunta:** ¿Existe relación entre población y emisiones totales de CO₂?

El gráfico de dispersión sirve para ver si dos variables numéricas se mueven juntas. Se usa escala logarítmica porque las diferencias entre países son muy grandes.

In [ ]:
relacion = paises_reciente.dropna(subset=["population", "co2", "co2_per_capita"]).copy()
relacion = relacion[relacion["population"] >= 1_000_000]

fig_dispersion = px.scatter(
    relacion,
    x="population",
    y="co2",
    size="co2_per_capita",
    color="co2_per_capita",
    hover_name="country",
    log_x=True,
    log_y=True,
    title=f"Relación entre población y emisiones de CO₂ en {anio_reciente}",
    labels={
        "population": "Población",
        "co2": "CO₂ total (millones de toneladas)",
        "co2_per_capita": "CO₂ per cápita"
    }
)

fig_dispersion.update_layout(template="plotly_white", title_x=0.02)
fig_dispersion.show()

In [ ]:
correlacion = relacion[["population", "co2"]].corr().iloc[0, 1]
print(f"Correlación entre población y CO₂ total en {anio_reciente}: {correlacion:.2f}")

**Interpretación:** se observa una relación positiva: países con mayor población tienden a registrar mayores emisiones totales. Sin embargo, el tamaño/color de los puntos permite ver que el CO₂ per cápita no depende solo de la población.

## 10. Visualización complementaria — Chile y comparación regional

Esta sección ayuda a conectar el análisis global con un caso cercano: Chile y países seleccionados de Latinoamérica.

In [ ]:
chile = paises[paises["country"] == "Chile"].dropna(subset=["co2"]).copy()
chile_1990 = chile[(chile["year"] >= 1990) & (chile["year"] <= anio_reciente)]

fig_chile = px.line(
    chile_1990,
    x="year",
    y="co2",
    markers=True,
    title=f"Evolución de emisiones de CO₂ en Chile, 1990-{anio_reciente}",
    labels={"year": "Año", "co2": "CO₂ total (millones de toneladas)"}
)
fig_chile.update_layout(template="plotly_white", title_x=0.02)
fig_chile.show()

In [ ]:
paises_latam = ["Chile", "Argentina", "Brazil", "Peru", "Colombia", "Mexico"]
latam_reciente = (
    paises_reciente[paises_reciente["country"].isin(paises_latam)]
    .dropna(subset=["co2"])
    .sort_values("co2", ascending=False)
)

fig_latam = px.bar(
    latam_reciente,
    x="country",
    y="co2",
    color="country",
    title=f"Comparación regional de CO₂ total en {anio_reciente}",
    labels={"country": "País", "co2": "CO₂ total (millones de toneladas)"},
    hover_data=["co2_per_capita", "population"]
)
fig_latam.update_layout(template="plotly_white", title_x=0.02, showlegend=False)
fig_latam.show()

**Interpretación:** Chile tiene emisiones totales menores que países de mayor población como Brasil o México. Sin embargo, la métrica per cápita permite complementar la lectura, porque compara las emisiones en relación con la población.

## 11. Mini dashboard final

El dashboard reúne indicadores y gráficos con una secuencia lógica:

1. Indicadores principales.
2. Tendencia mundial.
3. Comparación entre países.
4. Relación entre variables.
5. Caso Chile y comparación regional.

Esto evita mostrar gráficos sueltos sin propósito.

In [ ]:
display(HTML(html))
fig_lineas.show()
fig_barras.show()
fig_dispersion.show()
fig_chile.show()
fig_latam.show()

## 12. Conclusiones

1. Las emisiones mundiales de CO₂ muestran una tendencia general creciente desde 1990 hasta el año más reciente disponible.
2. Los mayores emisores totales se concentran en un grupo reducido de países.
3. Existe una relación positiva entre población y emisiones totales, aunque el CO₂ per cápita permite una lectura más justa entre países.
4. Chile presenta emisiones totales menores que países más poblados de la región, por lo que conviene mirar tanto CO₂ total como CO₂ per cápita.

## 13. Evidencia Git/GitHub

Comandos sugeridos para documentar el flujo de trabajo en el repositorio:

```bash
git status
git add .
git commit -m "Agrega analisis inicial de CO2"
git push
```

Después de editar el `README.md` desde GitHub web:

```bash
git pull
git log --oneline -3
```

**Evidencias recomendadas para la presentación:**

- Link del repositorio en GitHub.
- Captura del notebook o archivo Python visible en GitHub.
- Captura de `git log --oneline -3`.
- Captura de `git pull` ejecutado correctamente o repositorio actualizado.